# Automação de Indicadores de Desempenho de Lojas


## Introdução

Este notebook implementa um fluxo automatizado de **ETL, análise de metas, geração de relatórios visuais e disparo de e-mails executivos** para gerentes de lojas e para a diretoria da rede.

### Arquitetura e Fluxo de Execução
1. **Ambiente e Configurações:** Identifica se a execução ocorre em Jupyter Notebook ou script `.py` (com suporte a flags via CLI) e carrega parâmetros do `config.toml` e credenciais do `.env`.
2. **Processamento e Consolidação (ETL):** Lê bases brutas (`Vendas.xlsx`, `Metas.csv`, `Emails.xlsx`, `Lojas.csv`), alinha datas para a janela móvel dos últimos 365 dias, gera planilhas individuais formatadas por loja (`Diário` e `Mensal`) e consolida indicadores de faturamento, diversidade de produtos e ticket médio.
3. **Relatórios Visuais e Simulação PDF:** Renderiza templates HTML responsivos via **Jinja2** com CSS inline (**Premailer**), gera gráficos em alta resolução com **Plotly** e cria simulações fiéis das mensagens em PDF via **Playwright**.
4. **Disparo via SMTP:** Envia e-mails individuais para cada gerente de loja (com planilha anexa) e um resumo consolidado com rankings para a diretoria.

## 1. Importações e estruturas de dados

Carrega bibliotecas, estruturas tipadas e classes auxiliares usadas em todo o fluxo.


In [1]:
# ==============================================================================
# IMPORTAÇÕES
# ==============================================================================


import base64
import gc
import io
import logging
import smtplib
import sys
import tomllib
from collections.abc import Callable
from concurrent.futures import ThreadPoolExecutor
from email.message import EmailMessage
from email.utils import make_msgid
from pathlib import Path
from typing import Any, NamedTuple, TypedDict

import pandas as pd
import plotly.colors as pc
import plotly.graph_objects as go
from dotenv import dotenv_values
from jinja2 import Environment, FileSystemLoader, Template
from openpyxl.styles import Alignment, Font
from openpyxl.worksheet.worksheet import Worksheet
from plotly.subplots import make_subplots
from playwright.sync_api import sync_playwright
from premailer import transform


# ==============================================================================
# ESTRUTURAS DE DADOS E TIPAGEM
# ==============================================================================


class ConfigOutputs(TypedDict):
    exibir_prints_acompanhamento: bool
    manter_planilhas: bool
    exportar_graficos: bool
    gerar_pdfs_email: bool


class ConfigCaminhos(TypedDict):
    saida_planilhas: str
    saida_pdfs: str
    saida_graficos: str


class Configuracoes(TypedDict):
    outputs: ConfigOutputs
    caminhos: ConfigCaminhos


class CredenciaisSMTP(NamedTuple):
    remetente: str
    senha: str
    porta: str
    servidor: str
    destinatario_teste: str | None
    nome_remetente: str


class BasesProcessadas(NamedTuple):
    caminho_raw: Path
    df_emails: pd.DataFrame
    df_metas_lojas: pd.DataFrame
    df_vendas: pd.DataFrame
    dic_planilhas_lojas: dict[int, Path]


class MetasConsolidadas(NamedTuple):
    dic_metas_lojas: dict
    df_vendas_lojas: pd.DataFrame
    dic_metas_rede: dict
    df_vendas_rede: pd.DataFrame

## 2. Ambiente, Configurações, Logs e Credenciais

Funções responsáveis por preparar a infraestrutura da execução antes da leitura de dados:
* **Detecção de Ambiente:** Adapta caminhos relativos para notebooks em subpastas ou scripts em produção, habilitando argumentos de linha de comando (`argparse`) fora do Jupyter.
* **Rastreabilidade:** Configura logs diários em `logs/aaaa-mm-dd-log.txt` com saída simultânea no console.
* **Segurança:** Lê credenciais SMTP diretamente do arquivo `.env` a cada execução via `dotenv_values`, evitando cache residual na memória da sessão.

### 2.1. Identificação do ambiente de execução

Determina se o programa está no notebook ou em ambiente externo e localiza a raiz do projeto.


In [2]:
# ==============================================================================
# AMBIENTE, CONFIGURAÇÕES E LOGS
# ==============================================================================


def identificar_ambiente() -> tuple[bool | None, Path | None]:
    """Identifica o ambiente de execução e mapeia o caminho raiz do projeto.

    Returns:
        tuple[bool | None, Path | None]: Sinalizador de ambiente Jupyter Notebook (bool)
            e caminho raiz do projeto (Path). Havendo erro, retorna (None, None).
    """
    try:
        eh_notebook = "ipykernel" in sys.modules
        caminho_raiz = Path.cwd().parent if eh_notebook else Path(__file__).resolve().parent.parent
        return eh_notebook, caminho_raiz
    except Exception as erro:
        print(f"{pd.Timestamp.now().strftime('%H:%M:%S')} - Erro na identificação do ambiente: {erro}!")
        return None, None

### 2.2. Carregamento das configurações

Lê o arquivo de configurações e aplica opções de execução quando estiver fora do notebook.


In [3]:
def carregar_configuracoes(eh_notebook: bool, caminho_raiz: Path) -> Configuracoes:
    """Carrega as configurações do arquivo config.toml e aplica flags de terminal, se houver.

    Args:
        eh_notebook (bool): Indicador de execução em ambiente Jupyter Notebook.
        caminho_raiz (Path): Caminho raiz do projeto.

    Returns:
        Configuracoes: Dicionário estruturado contendo as chaves 'outputs' e 'caminhos'.
            Em caso de erro na leitura do arquivo, retorna os valores padrão.
    """
    try:
        with open(caminho_raiz / "config.toml", "rb") as arquivo_config:
            config = tomllib.load(arquivo_config)
    except Exception as erro:
        config = {
            "outputs": {
                "exibir_prints_acompanhamento": True,
                "manter_planilhas": True,
                "exportar_graficos": True,
                "gerar_pdfs_email": True,
            },
            "caminhos": {
                "saida_planilhas": "data/processed/backup_vendas_lojas/",
                "saida_pdfs": "reports/pdfs/",
                "saida_graficos": "reports/graficos/",
            },
        }
        print(f"{pd.Timestamp.now().strftime('%H:%M:%S')} - Erro no carregamento do config.toml: {erro}!")
        print("A execução ocorrerá com os valores padrão.")

    if not eh_notebook:
        import argparse

        parser = argparse.ArgumentParser(description="Automação de Indicadores de Lojas")
        parser.add_argument("-s", "--silencioso", action="store_true", help="Não exibe prints de acompanhamento")
        parser.add_argument("-m", "--nao-manter-planilhas", action="store_true", help="Remove planilhas geradas")
        parser.add_argument("-g", "--sem-graficos", action="store_true", help="Não exporta imagens dos gráficos")
        parser.add_argument("-p", "--sem-pdfs", action="store_true", help="Não gera PDFs de simulação dos e-mails")
        args = parser.parse_args()

        if args.silencioso:
            config["outputs"]["exibir_prints_acompanhamento"] = False
        if args.nao_manter_planilhas:
            config["outputs"]["manter_planilhas"] = False
        if args.sem_pdfs:
            config["outputs"]["gerar_pdfs_email"] = False
        if args.sem_graficos:
            config["outputs"]["exportar_graficos"] = False

    return config

### 2.3. Configuração dos logs

Define o arquivo de log do dia e o formato das mensagens registradas.


In [4]:
def configurar_log(caminho_raiz: Path, data_hoje_str_amd: str) -> bool:
    """Configura o arquivo de registro de logs da execução diária.

    Args:
        caminho_raiz (Path): Caminho raiz do projeto.
        data_hoje_str_amd (str): Data atual no formato aaaa-mm-dd.

    Returns:
        bool: True se configurado com sucesso, False em caso de erro.
    """
    try:
        caminho_logs = caminho_raiz / "logs"
        caminho_logs.mkdir(exist_ok=True)
        logging.basicConfig(
            filename=caminho_logs / f"{data_hoje_str_amd}-log.txt",
            level=logging.INFO,
            format="%(asctime)s - %(levelname)s - %(message)s",
            encoding="utf-8",
        )
        return True
    except Exception as erro:
        print(f"{pd.Timestamp.now().strftime('%H:%M:%S')} - Erro na configuração do log: {erro}!")
        return False

### 2.4. Registro de eventos

Centraliza o registro de informações e possíveis erros no console e no arquivo de log.


In [5]:
def registrar_log(mensagem: str, tipo: str = "INFO", exibir_prints: bool = True) -> bool:
    """Registra um evento no arquivo de log e, opcionalmente, o exibe no console.

    Args:
        mensagem (str): Descrição do evento a ser registrado.
        tipo (str): Nível do log ('INFO', 'DEBUG', 'WARNING', 'ERROR' ou 'CRITICAL').
        exibir_prints (bool): Indica se a mensagem deve ser impressa na tela.

    Returns:
        bool: True se registrado com sucesso, False em caso de erro.
    """
    try:
        horario = pd.Timestamp.now().strftime("%H:%M:%S")
        if tipo == "CRITICAL" or exibir_prints:
            print(f"{horario} - {mensagem}")

        funcao_log = getattr(logging, tipo.lower(), logging.info)
        funcao_log(mensagem)
        return True
    except Exception as erro:
        print(f"{pd.Timestamp.now().strftime('%H:%M:%S')} - Erro no registro de log: {erro}!")
        return False

### 2.5. Encerramento dos logs

Fecha e remove os manipuladores do arquivo de log ao final da execução.


In [6]:
def encerrar_log():
    """Fecha e remove todos os manipuladores de arquivo do sistema de logs."""
    logger = logging.getLogger()
    for handler in logger.handlers[:]:
        handler.close()
        logger.removeHandler(handler)

### 2.6. Leitura das credenciais de e-mail

Valida e recupera as variáveis necessárias para conectar ao servidor de e-mail.


In [7]:
def buscar_variaveis_ambiente(caminho_env: Path) -> CredenciaisSMTP | None:
    """Lê e valida as variáveis de ambiente diretamente do arquivo .env.

    Args:
        caminho_env (Path): Caminho completo do arquivo .env.

    Returns:
        CredenciaisSMTP | None: Objeto nomeado contendo os atributos:
            - remetente (str): Endereço de e-mail do remetente.
            - senha (str): Senha do e-mail do remetente.
            - porta (str): Porta SMTP do servidor.
            - servidor (str): Endereço do servidor SMTP.
            - destinatario_teste (str | None): E-mail de destinatário para testes.
            - nome_remetente (str): Nome de exibição do remetente.
            Retorna None em caso de erro ou ausência de variável obrigatória.
    """
    try:
        if not caminho_env.is_file():
            registrar_log(f"Arquivo de variáveis de ambiente ({caminho_env}) não encontrado!", "CRITICAL")
            return None

        env_vars = dotenv_values(caminho_env)
        var_obrigatorias = ("REMETENTE_EMAIL", "REMETENTE_SENHA", "PORTA_SMTP", "SERVIDOR_SMTP", "REMETENTE_NOME")

        var_validadas = {}
        for var in var_obrigatorias:
            valor = (env_vars.get(var) or "").strip()
            if not valor:
                registrar_log(f'Variável de ambiente "{var}" ausente ou vazia!', "CRITICAL")
                return None
            var_validadas[var] = valor

        destinatario_teste = (env_vars.get("DESTINATARIO_EMAIL_TESTE") or "").strip() or None

        return CredenciaisSMTP(
            remetente=var_validadas["REMETENTE_EMAIL"],
            senha=var_validadas["REMETENTE_SENHA"],
            porta=var_validadas["PORTA_SMTP"],
            servidor=var_validadas["SERVIDOR_SMTP"],
            destinatario_teste=destinatario_teste,
            nome_remetente=var_validadas["REMETENTE_NOME"],
        )
    except Exception as erro:
        registrar_log(f"Erro no acesso às variáveis de ambiente: {erro}!", "CRITICAL")
        return None

## 3. Tratamento de Dados e Formatação de Planilhas Excel

Camada de transformação dos dados financeiros e padronização visual das entregas em `.xlsx`:
* **Conversão Numérica:** Padroniza valores monetários entre o formato brasileiro/europeu (`1.234,56`) e ponto flutuante (`float`).
* **Estilização Centralizada (`_formatar_aba_excel`):** Aplica máscaras nativas do Excel (`dd/mm/yyyy`, `#,##0`, `#,##0.00`) e substitui indicadores booleanos por semáforos coloridos (`⬤`) via `openpyxl`.

### 3.1. Conversão de valores monetários

Padroniza valores em texto no formato monetário brasileiro para cálculos numéricos.


In [8]:
# ==============================================================================
# TRATAMENTO DE DADOS E PLANILHAS EXCEL
# ==============================================================================


def moeda_para_decimal(valor: Any) -> float | Any | None:
    """Converte strings monetárias no padrão brasileiro em números decimais (float).

    Args:
        valor (Any): Valor a ser avaliado e convertido.

    Returns:
        float | Any | None: Número decimal convertido, o próprio valor original se não
            for string, ou None em caso de erro de conversão.
    """
    try:
        if isinstance(valor, str):
            return float(valor.replace("R$", "").replace(".", "").replace(",", ".").strip())
        return valor
    except Exception as erro:
        registrar_log(f"Erro na conversão de moeda para decimal: {erro}!", "CRITICAL")
        return None

### 3.2. Formatação das planilhas Excel

Aplica máscaras, alinhamentos e semáforos nas abas geradas no Excel.


In [9]:
def _formatar_aba_excel(
    ws: Worksheet,
    df: pd.DataFrame,
    formatos_colunas: dict[str, str] | None = None,
    colunas_semaforo: list[str] | None = None,
) -> None:
    """Aplica máscaras de formatação numérica/data e semáforos coloridos em uma aba do Excel.

    Args:
        ws (Worksheet): Aba do openpyxl a ser formatada.
        df (pd.DataFrame): DataFrame correspondente aos dados da aba.
        formatos_colunas (dict[str, str] | None): Dicionário mapeando {nome_coluna: mascara_excel}.
        colunas_semaforo (list[str] | None): Lista de colunas booleanas a converter em círculos coloridos.
    """
    formatos_colunas = formatos_colunas or {}
    colunas_semaforo = colunas_semaforo or []

    fonte_verde = Font(name="Segoe UI Symbol", color="188038", size=14)
    fonte_vermelha = Font(name="Segoe UI Symbol", color="D93025", size=14)
    alinhamento_centro = Alignment(horizontal="center", vertical="center")

    for col_idx, col_nome in enumerate(df.columns, start=1):
        if col_nome in formatos_colunas:
            mascara = formatos_colunas[col_nome]
            for row in range(2, len(df) + 2):
                ws.cell(row=row, column=col_idx).number_format = mascara
        elif col_nome in colunas_semaforo:
            for row in range(2, len(df) + 2):
                celula = ws.cell(row=row, column=col_idx)
                atingiu = bool(celula.value)
                celula.value = "⬤"
                celula.font = fonte_verde if atingiu else fonte_vermelha
                celula.alignment = alinhamento_centro

## 4. Extração, Janela Móvel e Cálculo de Indicadores

Nesta etapa, os dados brutos são validados e processados:
* **Alinhamento Temporal:** Desloca a série histórica de vendas para que o registro mais recente corresponda exatamente ao dia anterior (`data_ontem`), filtrando os últimos 365 dias.
* **Exportação por Loja:** Gera arquivos Excel individuais com abas `Diário` e `Mensal` para anexo nos e-mails dos gerentes.
* **Apuração de Metas:** Calcula faturamento, diversidade de SKUs e ticket médio (diário e anual) por loja e para o consolidado da rede, comparando os resultados com as metas estabelecidas.

### 4.1. Preparação das bases e arquivos processados

Lê os arquivos brutos, ajusta as datas e cria as planilhas diárias e mensais das lojas.


In [10]:
def preparar_bases_processadas(
    caminho_raiz: Path,
    caminho_planilhas: Path,
    destinatario_teste: str | None,
    data_ontem: pd.Timestamp,
) -> BasesProcessadas | None:
    """Valida os arquivos de entrada, ajusta datas e gera as planilhas individuais das lojas.

    Args:
        caminho_raiz (Path): Caminho raiz do projeto.
        caminho_planilhas (Path): Diretório de saída para as planilhas processadas das lojas.
        destinatario_teste (str | None): E-mail destinatário para preenchimento de linhas vazias em testes.
        data_ontem (pd.Timestamp): Data de referência do dia anterior ao atual.

    Returns:
        BasesProcessadas | None: Objeto nomeado contendo os atributos:
            - caminho_raw (Path): Caminho do diretório de arquivos originais.
            - df_emails (pd.DataFrame): Tabela de e-mails com destinatários vazios preenchidos.
            - df_metas_lojas (pd.DataFrame): Tabela de metas das lojas com valores monetários convertidos.
            - df_vendas (pd.DataFrame): Tabela de vendas filtrada para os últimos 365 dias.
            - dic_planilhas_lojas (dict[int, Path]): Dicionário mapeando o ID de cada loja ao caminho da planilha.
            Retorna None em caso de erro ou ausência de arquivos obrigatórios.
    """
    try:
        caminho_raw = caminho_raiz / "data/raw"
        if not caminho_raw.is_dir():
            registrar_log(f'Diretório "data/raw" ("{caminho_raw}") não encontrado!', "CRITICAL")
            return None

        caminho_planilhas.mkdir(parents=True, exist_ok=True)

        caminho_emails = caminho_raw / "Emails.xlsx"
        if not caminho_emails.is_file():
            registrar_log(f'Arquivo "{caminho_emails.name}" não encontrado!', "CRITICAL")
            return None

        df_emails = pd.read_excel(caminho_emails, index_col="Loja", dtype={"E-mail": "string"})
        ser_sem_email = df_emails["E-mail"].isna() | (df_emails["E-mail"].str.strip() == "")
        if ser_sem_email.any():
            if destinatario_teste:
                df_emails.loc[ser_sem_email, "E-mail"] = destinatario_teste
            else:
                registrar_log(
                    'Planilha de e-mails possui linhas vazias e "DESTINATARIO_EMAIL_TESTE" não foi configurado!',
                    "CRITICAL",
                )
                return None

        caminho_metas = caminho_raw / "Metas.csv"
        if not caminho_metas.is_file():
            registrar_log(f'Arquivo "{caminho_metas.name}" não encontrado!', "CRITICAL")
            return None

        df_metas_lojas = pd.read_csv(caminho_metas, sep=";", encoding="cp1252", index_col=0)
        df_metas_lojas["Faturamento_Num"] = df_metas_lojas["Faturamento"].apply(moeda_para_decimal)
        df_metas_lojas["Ticket Médio_Num"] = df_metas_lojas["Ticket Médio"].apply(moeda_para_decimal)

        caminho_vendas = caminho_raw / "Vendas.xlsx"
        if not caminho_vendas.is_file():
            registrar_log(f'Arquivo "{caminho_vendas.name}" não encontrado!', "CRITICAL")
            return None

        df_vendas = pd.read_excel(caminho_vendas)
        df_vendas["Data"] = pd.to_datetime(df_vendas["Data"]).dt.normalize()

        qtd_dias_ajuste = (data_ontem - df_vendas["Data"].max()).days
        df_vendas["Data"] = df_vendas["Data"] + pd.to_timedelta(qtd_dias_ajuste, unit="D")

        data_corte_anual = data_ontem - pd.DateOffset(years=1) + pd.Timedelta(days=1)
        df_vendas = df_vendas[df_vendas["Data"] >= data_corte_anual]

        data_ontem_str_amd = data_ontem.strftime("%Y-%m-%d")
        dic_planilhas_lojas: dict[int, Path] = {}

        formatos_diario = {
            "Data": "dd/mm/yyyy",
            "Quantidade": "#,##0",
            "Valor Unitário": "#,##0.00",
            "Valor Final": "#,##0.00",
        }
        formatos_mensal = {
            "Mês": "mm/yyyy",
            "Quantidade": "#,##0",
            "Valor Final": "#,##0.00",
        }

        for id_loja, df_loja_dia in df_vendas.groupby("ID Loja"):
            caminho_proc_loja = caminho_planilhas / f"Loja_{id_loja}"
            caminho_proc_loja.mkdir(exist_ok=True)
            caminho_arq_loja = caminho_proc_loja / f"{data_ontem_str_amd}_Vendas_Loja_{id_loja}.xlsx"

            df_loja_dia = df_loja_dia.drop(columns=["ID Loja"])
            dic_planilhas_lojas[id_loja] = caminho_arq_loja

            df_loja_mes = (
                df_loja_dia.groupby(df_loja_dia["Data"].dt.to_period("M"))[["Quantidade", "Valor Final"]]
                .sum()
                .reset_index()
                .rename(columns={"Data": "Mês"})
            )
            df_loja_mes["Mês"] = df_loja_mes["Mês"].dt.to_timestamp()

            with pd.ExcelWriter(caminho_arq_loja, engine="openpyxl", datetime_format="mm/yyyy") as writer:
                df_loja_dia.to_excel(writer, sheet_name="Diário", index=False)
                df_loja_mes.to_excel(writer, sheet_name="Mensal", index=False)
                _formatar_aba_excel(writer.sheets["Diário"], df_loja_dia, formatos_colunas=formatos_diario)
                _formatar_aba_excel(writer.sheets["Mensal"], df_loja_mes, formatos_colunas=formatos_mensal)

        return BasesProcessadas(
            caminho_raw=caminho_raw,
            df_emails=df_emails,
            df_metas_lojas=df_metas_lojas,
            df_vendas=df_vendas.drop(columns=["Código Venda", "Valor Unitário"]),
            dic_planilhas_lojas=dic_planilhas_lojas,
        )
    except Exception as erro:
        registrar_log(f"Erro na preparação das bases processadas: {erro}!", "CRITICAL")
        return None

### 4.2. Cálculo das métricas por período

Calcula faturamento, diversidade de produtos e ticket médio para um período.


In [11]:
def _calcular_metricas_periodo(df: pd.DataFrame, agrupador: str | Callable, sufixo: str) -> pd.DataFrame:
    """Calcula soma do faturamento, quantidade de produtos distintos e ticket médio da tabela informada.

    Args:
        df (pd.DataFrame): Tabela a ter as métricas calculadas.
        agrupador (str | Callable): Coluna ou função de agrupamento.
        sufixo (str): Sufixo da coluna a ser criada na tabela.

    Returns:
        pd.DataFrame: Tabela com as métricas calculadas.
    """
    df_resumo = df.groupby(agrupador).agg(**{
            f"Fat_{sufixo}": pd.NamedAgg(column="Valor Final", aggfunc="sum"),
            f"Prod_Div_{sufixo}": pd.NamedAgg(column="Produto", aggfunc="nunique"),
            f"Qtd_{sufixo}": pd.NamedAgg(column="Quantidade", aggfunc="sum")
        }
    )
    df_resumo[f"Tic_Medio_{sufixo}"] = (df_resumo[f"Fat_{sufixo}"] / df_resumo[f"Qtd_{sufixo}"]).fillna(0)
    
    return df_resumo.drop(columns=[f"Qtd_{sufixo}"])

### 4.3. Consolidação das vendas por loja e rede

Consolida os indicadores anuais e do dia anterior por loja e para toda a rede.


In [12]:
def gerar_valores_consolidados(df_vendas: pd.DataFrame, data_ontem: pd.Timestamp) -> tuple[pd.DataFrame | None, pd.DataFrame | None]:
    """Gera tabelas anual e do dia anterior de valores acumulados de faturamento e quantidades de produtos distintos de venda e ticket médio por loja e de toda rede de lojas.

    Args:
        df_vendas (pd.DataFrame): Tabela geral de vendas.
        data_ontem (pd.Timestamp): Data do dia anterior.

    Returns:
        tuple: Tabela de valores consolidados por loja, do ano e do dia anterior ao atual (pd.DataFrame), e Tabela de valores consolidados de toda rede, do ano e do dia anterior ao atual (pd.DataFrame).
               Havendo erro, todos os itens retornam None.
    """
    try:
        df_filtrado_ontem = df_vendas[df_vendas["Data"] == data_ontem]

        def consolidar_por_nivel(agrupador: str|Callable) -> pd.DataFrame:
            """Consolida dados conforme o nível.

            Args:
                agrupador (str|Callable): Coluna ou função de agrupamento.

            Returns:
                pd.DataFrame: Tabela consolidada.
            """
            df_ano = _calcular_metricas_periodo(df_vendas, agrupador, "Ano")
            df_ontem = _calcular_metricas_periodo(df_filtrado_ontem, agrupador, "Ontem")

            df_consolidado = df_ano.join(df_ontem).fillna(0)
            colunas_inteiras = ["Prod_Div_Ano", "Prod_Div_Ontem"]
            df_consolidado[colunas_inteiras] = df_consolidado[colunas_inteiras].astype(int)
            
            return df_consolidado

        # Gera os dois DataFrames usando a mesma lógica
        df_vendas_consolidadas_lojas = consolidar_por_nivel("ID Loja")
        df_vendas_consolidadas_rede = consolidar_por_nivel(lambda _: "Total Rede")


        return df_vendas_consolidadas_lojas, df_vendas_consolidadas_rede
    
    except Exception as erro:
        txt_erro = f"Erro na geração de valores acumulados: {erro}!"
        registrar_log(txt_erro, "CRITICAL")
        return None, None

### 4.4. Unificação dos dados cadastrais das lojas

Une os dados de lojas e e-mails em uma tabela principal.


In [13]:
def unificar_dados_lojas(caminho_arquivo_lojas: Path, df_emails: pd.DataFrame) -> tuple[pd.DataFrame | None, dict | None, str | None]:
    """Unifica os dados gerais das lojas em uma única tabela.

    Args:
        caminho_arquivo_lojas (str): Caminho do arquivo geral de lojas.
        df_emails (pd.DataFrame): Tabela de e-mails.

    Returns:
        tuple[pd.DataFrame | None, dict[int, str] | None, str | None]: Tabela contendo os dados gerais das lojas,
            dicionário mapeando {id_loja: nome_loja} e o e-mail da diretoria. Havendo erro, todos retornam None.
    """
    try:
        if not caminho_arquivo_lojas.is_file():
            registrar_log(f'Arquivo "{caminho_arquivo_lojas.name}" ("{caminho_arquivo_lojas}") não encontrado!', "CRITICAL")
            return None, None, None
        df_lojas = pd.read_csv(caminho_arquivo_lojas, sep=";", encoding="cp1252", index_col="Loja")
        df_lojas = df_lojas.join(df_emails[["Gerente", "E-mail"]])
        df_lojas = df_lojas.reset_index()
        df_lojas = df_lojas.set_index("ID Loja")
        email_diretoria = df_emails.loc["Diretoria", "E-mail"]
        del df_emails
        gc.collect()
        dic_nomes_lojas = dict(zip(df_lojas.index, df_lojas["Loja"]))
        
        return df_lojas, dic_nomes_lojas, email_diretoria
    
    except Exception as erro:
        txt_erro = f"Erro na unificação de dados gerais das lojas: {erro}!"
        registrar_log(txt_erro, "CRITICAL")
        return None, None, None

### 4.5. Aplicação de metas e organização das tabelas

Compara os indicadores com as metas e organiza a ordem das colunas.


In [14]:
def _aplicar_metas_e_ordenar(df: pd.DataFrame, dic_metas: dict) -> pd.DataFrame:
    """Cria colunas booleanas de atingimento de metas e organiza a ordem das colunas.

    Args:
        df (pd.DataFrame): Tabela consolidada de indicadores realizados.
        dic_metas (dict): Dicionário contendo as metas correspondentes ao nível da tabela.

    Returns:
        pd.DataFrame: Tabela atualizada com indicadores intercalados por seus status de meta.
    """
    regras_metas = [
        ("Meta_Fat_Ontem", "Fat_Ontem", "fat_dia"),
        ("Meta_Div_Ontem", "Prod_Div_Ontem", "div_dia"),
        ("Meta_Tic_Ontem", "Tic_Medio_Ontem", "tic_dia"),
        ("Meta_Fat_Ano", "Fat_Ano", "fat_ano"),
        ("Meta_Div_Ano", "Prod_Div_Ano", "div_ano"),
        ("Meta_Tic_Ano", "Tic_Medio_Ano", "tic_ano"),
    ]

    novas_colunas = {
        col_meta: df[col_valor] >= dic_metas[chave_meta]
        for col_meta, col_valor, chave_meta in regras_metas
    }

    ordem_colunas = []
    for col_meta, col_valor, _ in regras_metas:
        ordem_colunas.extend([col_valor, col_meta])

    return df.assign(**novas_colunas)[ordem_colunas]

### 4.6. Cálculo e avaliação de atingimento de metas

Consolida os parâmetros de metas e verifica o desempenho individual das **lojas** e da **rede** como um todo.


In [15]:
def preencher_atingimento_metas(
    df_metas_lojas: pd.DataFrame,
    df_vendas_consolidadas_lojas: pd.DataFrame,
    df_vendas_consolidadas_rede: pd.DataFrame,
) -> MetasConsolidadas | None:
    """Calcula metas da rede e avalia o atingimento das metas para lojas e rede.

    Args:
        df_metas_lojas (pd.DataFrame): Tabela de metas individuais das lojas.
        df_vendas_consolidadas_lojas (pd.DataFrame): Tabela de vendas consolidadas por loja.
        df_vendas_consolidadas_rede (pd.DataFrame): Tabela de vendas consolidadas de toda a rede.

    Returns:
        MetasConsolidadas | None: Objeto nomeado contendo os atributos:
            - dic_metas_lojas (dict): Dicionário com as metas diárias e anuais das lojas.
            - df_vendas_lojas (pd.DataFrame): Tabela consolidada das lojas com status de metas.
            - dic_metas_rede (dict): Dicionário com as metas diárias e anuais da rede.
            - df_vendas_rede (pd.DataFrame): Tabela consolidada da rede com status de metas.
            Retorna None em caso de erro.
    """
    try:
        dic_metas_lojas = {
            "fat_dia": df_metas_lojas.loc["Diário", "Faturamento_Num"],
            "div_dia": df_metas_lojas.loc["Diário", "Diversidade de Produtos"],
            "tic_dia": df_metas_lojas.loc["Diário", "Ticket Médio_Num"],
            "fat_ano": df_metas_lojas.loc["Anual", "Faturamento_Num"],
            "div_ano": df_metas_lojas.loc["Anual", "Diversidade de Produtos"],
            "tic_ano": df_metas_lojas.loc["Anual", "Ticket Médio_Num"],
        }

        qtd_lojas = len(df_vendas_consolidadas_lojas)
        dic_metas_rede = {
            chave: valor if chave in {"tic_dia", "tic_ano"} else valor * qtd_lojas
            for chave, valor in dic_metas_lojas.items()
        }

        return MetasConsolidadas(
            dic_metas_lojas=dic_metas_lojas,
            df_vendas_lojas=_aplicar_metas_e_ordenar(df_vendas_consolidadas_lojas, dic_metas_lojas),
            dic_metas_rede=dic_metas_rede,
            df_vendas_rede=_aplicar_metas_e_ordenar(df_vendas_consolidadas_rede, dic_metas_rede),
        )
    except Exception as erro:
        registrar_log(f"Erro no preenchimento do atingimento de metas: {erro}!", "CRITICAL")
        return None

## 5. Renderização HTML, Gráficos e Simulação em PDF

Camada de apresentação visual dos relatórios executivos (*OnePage*):
* **Templates e CSS Inline:** Compila os arquivos HTML via **Jinja2**, injeta tabelas de indicadores com semáforos (`🟢` / `🔴`), converte estilos para inline com **Premailer** (garantindo compatibilidade com clientes de e-mail) e gera Content-IDs (`cid:`) únicos no padrão RFC 5322.
* **Visualização de Dados:** Constrói gráficos de barras agrupadas (evolução mensal de faturamento e volume) e barras horizontais (ranking anual das lojas) exportados em PNG de alta definição.
* **Simulação PDF:** Utiliza **Playwright** (Chromium *headless*) para gerar arquivos PDF fiéis ao e-mail enviado, incluindo cartão visual do anexo. No Jupyter, a renderização roda em `ThreadPoolExecutor` para evitar conflitos com o loop assíncrono (`asyncio`) do kernel.

### 5.1. Configuração dos templates Jinja2

Valida os templates HTML e configura o ambiente de renderização Jinja2.


In [16]:
# ==============================================================================
# HTML, GRÁFICOS E PDF
# ==============================================================================


def configurar_ambiente_jinja(caminho_raiz: Path) -> tuple[Environment | None, str | None, str | None]:
    """Valida a existência dos templates HTML e configura o ambiente Jinja2.

    Args:
        caminho_raiz (Path): Caminho raiz do projeto.

    Returns:
        tuple[Environment | None, str | None, str | None]: Ambiente Jinja2 configurado,
            nome do template das lojas e nome do template da diretoria.
            Retorna (None, None, None) em caso de erro.
    """
    try:
        caminho_templates = caminho_raiz / "assets" / "templates"
        if not caminho_templates.is_dir():
            registrar_log(f'Diretório de templates ("{caminho_templates}") não encontrado!', "CRITICAL")
            return None, None, None

        nome_template_lojas = "template_email_lojas.html"
        nome_template_diretoria = "template_email_diretoria.html"

        for nome_tpl in (nome_template_lojas, nome_template_diretoria):
            if not (caminho_templates / nome_tpl).is_file():
                registrar_log(f'Template "{nome_tpl}" não encontrado em "{caminho_templates}"!', "CRITICAL")
                return None, None, None

        return Environment(loader=FileSystemLoader(caminho_templates)), nome_template_lojas, nome_template_diretoria
    except Exception as erro:
        registrar_log(f"Erro ao configurar o ambiente Jinja2: {erro}!", "CRITICAL")
        return None, None, None

### 5.2. Conversão para notação monetária europeia

Converte números para o padrão utilizado nos textos dos e-mails.


In [17]:
def converter_notacao_europeia(nro_notacao_anglicana: float) -> str:
    """Converte o fracionário informado em notação anglo-americana para notação europeia e 2 decimais.

    Args:
        nro_notacao_anglicana (float): Fracionário em notação anglo-americana.

    Returns:
        str: Texto com número em notação européia e 2 decimais.
    """
    return f"{nro_notacao_anglicana:,.2f}".replace(",", "_").replace(".", ",").replace("_", ".")

### 5.3. Construção das tabelas de indicadores

Monta as tabelas HTML com resultados, metas e semáforos.


In [18]:
def _gerar_tabelas_indicadores_html(colunas_dados: pd.Series, dic_metas: dict) -> tuple[str, str]:
    """Constrói as tabelas HTML diária e anual com indicadores, metas e semáforos.

    Args:
        colunas_dados (pd.Series): Série contendo os indicadores realizados da loja ou rede.
        dic_metas (dict): Dicionário com as metas correspondentes.

    Returns:
        tuple[str, str]: Strings HTML das tabelas de desempenho diário e anual.
    """
    def montar_tabela(sufixo_col: str, sufixo_meta: str, titulo_periodo: str) -> str:
        df_visual = pd.DataFrame(
            {
                f"Valor {titulo_periodo} (R$)": [
                    int(colunas_dados[f"Prod_Div_{sufixo_col}"]),
                    converter_notacao_europeia(colunas_dados[f"Fat_{sufixo_col}"]),
                    converter_notacao_europeia(colunas_dados[f"Tic_Medio_{sufixo_col}"]),
                ],
                f"Meta {titulo_periodo} (R$)": [
                    int(dic_metas[f"div_{sufixo_meta}"]),
                    converter_notacao_europeia(dic_metas[f"fat_{sufixo_meta}"]),
                    converter_notacao_europeia(dic_metas[f"tic_{sufixo_meta}"]),
                ],
                f"Cenário {titulo_periodo}": [
                    "🟢" if colunas_dados[f"Meta_Div_{sufixo_col}"] else "🔴",
                    "🟢" if colunas_dados[f"Meta_Fat_{sufixo_col}"] else "🔴",
                    "🟢" if colunas_dados[f"Meta_Tic_{sufixo_col}"] else "🔴",
                ],
            },
            index=["Diversidade de Produtos", "Faturamento", "Ticket Médio"],
        )
        return df_visual.to_html(border=0)

    return montar_tabela("Ontem", "dia", "Dia"), montar_tabela("Ano", "ano", "Ano")

### 5.4. Renderização dos e-mails em HTML

Compila o template, aplica estilos inline e prepara as imagens referenciadas.


In [19]:
def construir_html_email(
    nome_remetente: str,
    data_ontem_str_dma: str,
    colunas_dados: pd.Series,
    dic_metas: dict,
    template: Template,
    lst_cids_imagens: list[str],
    id_log: str,
    **dados_extras,
) -> tuple[str | None, dict[str, str] | None]:
    """Renderiza o template HTML com os dados, converte estilos para inline e mapeia CIDs.

    Args:
        nome_remetente (str): Nome do remetente da mensagem.
        data_ontem_str_dma (str): Data do dia anterior no formato dd/mm/aaaa.
        colunas_dados (pd.Series): Linha de dados consolidados da loja ou rede.
        dic_metas (dict): Dicionário de metas da loja ou rede.
        template (Template): Objeto Template compilado do Jinja2.
        lst_cids_imagens (list[str]): Lista de identificadores 'cid:' presentes no template.
        id_log (str): Identificador para registro de log em caso de falha.
        **dados_extras: Variáveis adicionais específicas de cada template.

    Returns:
        tuple[str | None, dict[str, str] | None]: HTML final com CSS inline e dicionário
            mapeando {cid_original: cid_gerado_sem_colchetes}. Retorna (None, None) em caso de erro.
    """
    try:
        html_dia, html_ano = _gerar_tabelas_indicadores_html(colunas_dados, dic_metas)
        html_corpo = template.render(
            data_ontem=data_ontem_str_dma,
            indicadores_dia=html_dia,
            indicadores_ano=html_ano,
            nome_remetente=nome_remetente,
            **dados_extras,
        )
        html_corpo = transform(html_corpo)

        dic_mapa_cids = {}
        for cid_original in (lst_cids_imagens or []):
            cid_sem_colch = make_msgid(domain="controladoria")[1:-1]
            html_corpo = html_corpo.replace(f"cid:{cid_original}", f"cid:{cid_sem_colch}")
            dic_mapa_cids[cid_original] = cid_sem_colch

        return html_corpo, dic_mapa_cids
    except Exception as erro:
        registrar_log(f"Erro ao renderizar o template HTML para {id_log}: {erro}!", "CRITICAL")
        return None, None

### 5.5. Gráfico mensal de evolução das vendas

Gera a evolução mensal de faturamento e quantidade como imagem.


In [20]:
def construir_grafico_evol_vendas(
    loja_id_nome: str | None,
    meses: pd.Series,
    quantidades: pd.Series,
    faturamentos: pd.Series,
    caminho_grafico: Path | None,
    id_log: str,
) -> bytes | None:
    """Gera o gráfico de evolução mensal (faturamento e quantidade) em bytes PNG.

    Args:
        loja_id_nome (str | None): Identificação 'ID - Nome' da loja, ou None para a rede.
        meses (pd.Series): Série temporal dos meses.
        quantidades (pd.Series): Série de quantidades vendidas por mês.
        faturamentos (pd.Series): Série de faturamento por mês.
        caminho_grafico (Path | None): Caminho para salvar a imagem em disco, se configurado.
        id_log (str): Identificador para registro de log em caso de erro.

    Returns:
        bytes | None: Imagem do gráfico em bytes PNG, ou None em caso de erro.
    """
    try:
        grafico_mensal = make_subplots(specs=[[{"secondary_y": True}]])
        grafico_mensal.add_trace(
            go.Bar(x=meses, y=faturamentos, name="Faturamento (R$)", offsetgroup=1),
            secondary_y=False,
        )
        grafico_mensal.add_trace(
            go.Bar(x=meses, y=quantidades, name="Quantidade de Produtos", offsetgroup=2),
            secondary_y=True,
        )

        titulo = (
            f"Evolução das Vendas Mensais da Loja {loja_id_nome}"
            if loja_id_nome is not None
            else "Evolução das Vendas Mensais da Rede de Lojas"
        )
        grafico_mensal.update_layout(
            template="plotly_dark",
            barmode="group",
            title=dict(text=titulo, x=0.5, xanchor="center"),
            margin=dict(t=85),
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
        )
        grafico_mensal.update_xaxes(ticklabelstandoff=15, tickangle=-45, automargin=True)
        grafico_mensal.update_yaxes(title_text="Faturamento (R$)", secondary_y=False)
        grafico_mensal.update_yaxes(title_text="Quantidade de Produtos", secondary_y=True)

        img_bytes = grafico_mensal.to_image(format="png", width=650, height=380, scale=2)
        if caminho_grafico:
            caminho_grafico.write_bytes(img_bytes)

        return img_bytes
    except Exception as erro:
        registrar_log(f"Erro ao gerar o gráfico de evolução mensal para {id_log}: {erro}!", "CRITICAL")
        return None

### 5.6. Gráfico anual de faturamento por loja

Gera o gráfico comparativo anual das lojas.


In [21]:
def construir_grafico_anual_lojas(df_lojas_ano: pd.DataFrame, caminho_grafico: Path | None) -> bytes | None:
    """Gera o gráfico de barras horizontais do faturamento anual por loja em bytes PNG.

    Args:
        df_lojas_ano (pd.DataFrame): Tabela ordenada contendo as colunas 'Loja' e 'Fat_Ano'.
        caminho_grafico (Path | None): Caminho para salvar a imagem em disco, se configurado.

    Returns:
        bytes | None: Imagem do gráfico em bytes PNG, ou None em caso de erro.
    """
    try:
        paleta_base = pc.qualitative.Pastel + pc.qualitative.Set3 + pc.qualitative.Bold
        cores_barras = [paleta_base[i % len(paleta_base)] for i in range(len(df_lojas_ano))]
        nomes_dentro_barras = [f"<i>{loja}</i>" for loja in df_lojas_ano["Loja"]]
        rotulos_valores_esquerda = [f"R$ {converter_notacao_europeia(valor)} " for valor in df_lojas_ano["Fat_Ano"]]

        grafico_anual = go.Figure(
            data=[
                go.Bar(
                    x=df_lojas_ano["Fat_Ano"],
                    y=df_lojas_ano["Loja"],
                    orientation="h",
                    marker_color=cores_barras,
                    text=nomes_dentro_barras,
                    textposition="inside",
                    insidetextanchor="start",
                    textfont=dict(color="black", size=12),
                    constraintext="none",
                )
            ]
        )

        maior_valor = df_lojas_ano["Fat_Ano"].max()
        grafico_anual.update_layout(
            template="plotly_dark",
            title=dict(text="Faturamento Acumulado Anual por Loja", x=0.5, xanchor="center"),
            margin=dict(l=20, r=30, t=60, b=30),
        )
        grafico_anual.update_xaxes(
            showticklabels=False,
            showgrid=False,
            zeroline=False,
            range=[0, maior_valor * 1.05 if maior_valor > 0 else 1],
        )
        grafico_anual.update_yaxes(
            tickmode="array",
            tickvals=df_lojas_ano["Loja"],
            ticktext=rotulos_valores_esquerda,
            tickfont=dict(family="Consolas, monospace", size=12, color="white"),
            automargin=True,
            showgrid=False,
        )

        altura_grafico = max(380, len(df_lojas_ano) * 28)
        img_bytes = grafico_anual.to_image(format="png", width=650, height=altura_grafico, scale=2)
        if caminho_grafico:
            caminho_grafico.write_bytes(img_bytes)

        return img_bytes
    except Exception as erro:
        registrar_log(f"Erro ao gerar o gráfico de faturamento anual por loja: {erro}!", "CRITICAL")
        return None

### 5.7. Renderização de PDFs com Playwright

Converte o conteúdo HTML em PDF por meio de uma página Chromium.


In [22]:
def _renderizar_pdf_playwright(msg_loja: bool, html_para_pdf: str, caminho_pdf: Path) -> None:
    """Converte o HTML recebido em PDF usando Chromium headless e o grava localmente.

    Args:
        msg_loja (bool): Sinalizador de mensagem enviada às lojas (ajusta para página única).
        html_para_pdf (str): HTML completo do corpo do e-mail com imagens em Base64.
        caminho_pdf (Path): Caminho completo do arquivo PDF a ser gerado.
    """
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page(viewport={"width": 750, "height": 800})
        page.set_content(html_para_pdf, wait_until="networkidle")

        altura_conteudo = page.evaluate("document.documentElement.scrollHeight")
        altura_maxima_a4 = 1060

        if msg_loja:
            escala = max(0.1, round(altura_maxima_a4 / altura_conteudo, 2)) if altura_conteudo > altura_maxima_a4 else 1.0
            margem = {"top": "4mm", "bottom": "4mm", "left": "10mm", "right": "10mm"}
        else:
            escala = 1.0
            margem = {"top": "12mm", "bottom": "12mm", "left": "12mm", "right": "12mm"}

        page.pdf(
            path=str(caminho_pdf),
            format="A4",
            print_background=True,
            scale=escala,
            margin=margem,
        )
        browser.close()

### 5.8. Geração da simulação em PDF

Insere gráficos e informações do anexo antes da geração do PDF.


In [23]:
def gerar_pdf_do_email(
    msg_loja: bool,
    html_email: str,
    dic_imagens_embutidas: dict[str, bytes],
    eh_notebook: bool,
    caminho_pdf: Path,
    exibir_prints: bool,
    nome_anexo: str,
    bytes_anexo: bytes,
) -> None:
    """Simula o HTML do e-mail em arquivo PDF, suportando gráficos embutidos e cartão visual de anexo.

    Args:
        msg_loja (bool): Sinalizador de mensagem enviada às lojas.
        html_email (str): HTML do corpo do e-mail.
        dic_imagens_embutidas (dict[str, bytes]): Dicionário mapeando {cid_sem_colchetes: img_bytes}.
        eh_notebook (bool): Sinalizador de execução em ambiente Jupyter Notebook.
        caminho_pdf (Path): Caminho completo do arquivo PDF a ser gerado.
        exibir_prints (bool): Sinalizador para exibição ou não das mensagens informativas.
        nome_anexo (str): Nome do arquivo anexado para exibição visual no PDF.
        bytes_anexo (bytes): Conteúdo em bytes do anexo para cálculo do tamanho.
    """
    try:
        html_para_pdf = html_email
        for cid_s_colch, img_bytes in dic_imagens_embutidas.items():
            img_base64 = base64.b64encode(img_bytes).decode("utf-8")
            html_para_pdf = html_para_pdf.replace(f"cid:{cid_s_colch}", f"data:image/png;base64,{img_base64}")

        if nome_anexo and bytes_anexo:
            tamanho_kb = max(1, round(len(bytes_anexo) / 1024))
            bloco_anexo_html = (
                '<div style="display: inline-flex; align-items: center; border: 1px solid #dadce0; '
                'border-radius: 4px; padding: 8px 12px; margin-top: 24px; background-color: #f8f9fa; '
                'font-family: Arial, sans-serif; min-width: 240px; page-break-inside: avoid;">'
                '<div style="width: 24px; height: 24px; background-color: #188038; color: #ffffff; '
                'font-weight: bold; font-size: 13px; display: flex; align-items: center; '
                'justify-content: center; border-radius: 3px; margin-right: 12px;">X</div>'
                '<div style="line-height: 1.3;">'
                f'<div style="font-size: 12px; font-weight: bold; color: #202124;">{nome_anexo}</div>'
                f'<div style="font-size: 11px; color: #5f6368;">{tamanho_kb}K</div>'
                "</div>"
                "</div>"
            )
            html_para_pdf = html_para_pdf.replace("</body>", f"{bloco_anexo_html}\n</body>")

        if eh_notebook:
            with ThreadPoolExecutor(max_workers=1) as executor:
                futuro = executor.submit(_renderizar_pdf_playwright, msg_loja, html_para_pdf, caminho_pdf)
                futuro.result()
        else:
            _renderizar_pdf_playwright(msg_loja, html_para_pdf, caminho_pdf)
    except Exception as erro:
        registrar_log(f"Falha ao gerar PDF {caminho_pdf.name}: {erro}", "ERROR", exibir_prints)

## 6. Comunicação SMTP e Disparo de Relatórios

Gerencia a autenticação segura (TLS) com o servidor de e-mail e realiza os envios:

* **Empacotamento MIME:** Constrói mensagens `EmailMessage` combinando alternativa HTML, imagens embutidas relacionadas (`add_related`) e planilhas Excel anexadas (`add_attachment`).
* **Fluxo Unificado (`_processar_pdf_e_enviar_email`):** Centraliza o mapeamento de CIDs, a geração opcional do PDF de conferência e o envio final tanto para os gerentes de loja quanto para a diretoria.

### 6.1. Conexão com o servidor SMTP

Estabelece uma conexão autenticada e segura com o servidor SMTP.


In [24]:
# ==============================================================================
# CONEXÃO SMTP E ENVIO DE E-MAILS
# ==============================================================================


def conectar_servidor_smtp(email_remetente: str, senha_email: str, porta_smtp: str, servidor_smtp: str) -> smtplib.SMTP | None:
    """Abre conexão segura (TLS) e realiza autenticação no servidor SMTP.

    Args:
        email_remetente (str): Endereço de e-mail da conta remetente.
        senha_email (str): Senha ou senha de aplicativo da conta remetente.
        porta_smtp (str): Porta de comunicação do servidor SMTP.
        servidor_smtp (str): Endereço do host SMTP.

    Returns:
        smtplib.SMTP | None: Objeto de conexão SMTP autenticado, ou None em caso de falha.
    """
    try:
        servidor_email = smtplib.SMTP(servidor_smtp, int(porta_smtp))
        servidor_email.starttls()
        servidor_email.login(email_remetente, senha_email)
        return servidor_email
    except Exception as erro:
        registrar_log(f"Erro na conexão com o servidor SMTP: {erro}!", "CRITICAL")
        return None

### 6.2. Encerramento da conexão SMTP

Fecha a sessão SMTP ao concluir o processo.


In [25]:
def desconectar_servidor_smtp(servidor_email: smtplib.SMTP, exibir_prints: bool):
    """Encerra a conexão ativa com o servidor SMTP.

    Args:
        servidor_email (smtplib.SMTP): Instância conectada do servidor SMTP.
        exibir_prints (bool): Indica se eventuais avisos devem ser exibidos no console.
    """
    try:
        servidor_email.quit()
        
    except Exception as erro:
        txt_erro = f"Erro na desconexão do servidor SMTP: {erro}!"
        registrar_log(txt_erro, "WARNING", exibir_prints)

### 6.3. Geração do anexo Excel em memória

Converte as tabelas de ranking em um arquivo Excel (`.xlsx`) formatado em memória para envio à diretoria.

In [26]:
def _converter_df_bytes(dic_abas: dict[str, pd.DataFrame]) -> bytes:
    """Converte um dicionário de DataFrames em bytes de abas formatadas no formato Excel (.xlsx).

    Args:
        dic_abas (dict[str, pd.DataFrame]): Dicionário mapeando {nome_aba: dataframe}.

    Returns:
        bytes: Conteúdo binário do arquivo Excel gerado.
    """
    buffer = io.BytesIO()

    formatos_diretoria = {
        "Faturamento (R$)": "#,##0.00",
        "Meta Faturamento (R$)": "#,##0.00",
        "Diversidade de Produtos": "#,##0",
        "Meta Diversidade": "#,##0",
        "Ticket Médio (R$)": "#,##0.00",
        "Meta Ticket (R$)": "#,##0.00",
    }
    semaforos_diretoria = ["Faturamento Ok", "Diversidade Ok", "Ticket Ok"]

    with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
        for aba, df in dic_abas.items():
            df.to_excel(writer, index=False, sheet_name=aba)
            _formatar_aba_excel(
                writer.sheets[aba],
                df,
                formatos_colunas=formatos_diretoria,
                colunas_semaforo=semaforos_diretoria,
            )

    return buffer.getvalue()

### 6.4. Envio das mensagens por SMTP

Monta a mensagem MIME, embute os gráficos, anexa a planilha e realiza o envio.

In [27]:
def enviar_email(
        servidor_email: smtplib.SMTP, 
        assunto: str, 
        remetente: str, 
        destinatario: str, 
        html_corpo: str, 
        dic_imagens_embutidas: dict[str, bytes], 
        caminho_planilha: Path | None, 
        bytes_anexo: bytes, 
        nome_anexo: str, 
        manter_planilhas: bool, 
        exibir_prints: bool, 
        id_log: str,) -> bool:
    """ Envia e-mails às lojas com respectivo desempenho no dia anterior e no consolidado do ano e planilha de vendas.

    Args:
        servidor_email (smtplib.SMTP): Servidor de e-mail a ser utilizado para o envio da mensagem.
        assunto (str): Assunto da mensagem.
        remetente (str): Endereço de e-mail do remetente da mensagem.
        destinatario (str): Endereço de e-mail do destinatário da mensagem.
        html_corpo (str): HTML do corpo da mensagem.
        dic_imagens_embutidas (dict[str, bytes]): Dicionário mapeando {cid_com_colchetes: img_bytes}.
        caminho_planilha (Path | None): Caminho da planilha de loja a ser anexada na mensagem.
        bytes_anexo (bytes): Arquivo em bytes para ser anexada na mensagem.
        nome_anexo (str): Nome do arquivo que será anexado na mensagem.
        manter_planilhas (bool): Indicador de manutenção ou não da planilha a ser anexada na mensagem em armazenamento local.
        exibir_prints (bool): Indicador para exibição ou não de mensagens informativas durante a execução.
        id_log (str): Texto identificador a ser registrado no arquivo de log em caso de erro.

    Returns:
        bool: Sucesso ou não da operação.
    """
    try:
        # Monta o pacote da mensagem (MIMEMultipart)
        msg = EmailMessage()
        msg["Subject"] = assunto
        msg["From"] = remetente
        msg["To"] = destinatario
        
        #  Define texto cru e HTML do corpo da mensagem
        msg.set_content("Seu leitor de e-mail não suporta visualização em HTML.")
        msg.add_alternative(html_corpo, subtype="html")
        
        # Percorre todos os gráficos informados (1 para lojas, 2 para diretoria) e embute no corpo da mensagem
        for cid_com_colch, img_bytes in dic_imagens_embutidas.items():
            msg.get_payload()[1].add_related(img_bytes, maintype="image", subtype="png", cid = cid_com_colch)
        
        # Coloca o anexo na mensagem
        msg.add_attachment(bytes_anexo, maintype="application", subtype="vnd.openxmlformats-officedocument.spreadsheetml.sheet", filename=nome_anexo)
        
        if caminho_planilha and not manter_planilhas:
            # Remove o vínculo da planilha no sistema operacional
            caminho_planilha.unlink(missing_ok=True)
        
        # Envia a mensagem
        servidor_email.send_message(msg)
        registrar_log(f"E-mail para {id_log} enviado com sucesso!", "INFO", exibir_prints)
        
        return True
    
    except Exception as erro:
        registrar_log(f"Falha ao enviar e-mail para {id_log}: {erro}", "CRITICAL")
        return False

### 6.5. Coordenação do PDF e do envio

Gera o PDF opcional e encaminha o e-mail com as imagens e a planilha corretas.


In [28]:
def _processar_pdf_e_enviar_email(
    servidor_email: smtplib.SMTP,
    assunto: str,
    destinatario: str,
    html_corpo: str,
    dic_mapa_cids: dict[str, str],
    dic_imagens_originais: dict[str, bytes],
    caminho_pdf: Path | None,
    eh_notebook: bool,
    msg_loja: bool,
    caminho_planilha: Path | None,
    bytes_anexo: bytes,
    nome_anexo_email: str,
    nome_anexo_pdf: str,
    manter_planilhas: bool,
    exibir_prints: bool,
    id_log: str,
) -> bool:
    """Mapeia os CIDs das imagens, gera o PDF opcional de simulação e envia o e-mail.

    Args:
        servidor_email (smtplib.SMTP): Servidor SMTP autenticado.
        assunto (str): Assunto do e-mail.
        destinatario (str): Endereço de e-mail do destinatário.
        html_corpo (str): Corpo HTML renderizado.
        dic_mapa_cids (dict[str, str]): Mapeamento {cid_original: cid_sem_colchetes}.
        dic_imagens_originais (dict[str, bytes]): Mapeamento {cid_original: img_bytes}.
        caminho_pdf (Path | None): Caminho para salvar o PDF de simulação, ou None.
        eh_notebook (bool): Indicador de ambiente Jupyter Notebook.
        msg_loja (bool): Indica se o layout do PDF deve ser ajustado para loja.
        caminho_planilha (Path | None): Caminho físico da planilha da loja, se houver.
        bytes_anexo (bytes): Conteúdo binário do anexo Excel.
        nome_anexo_email (str): Nome do anexo enviado no e-mail.
        nome_anexo_pdf (str): Nome do anexo exibido no cartão visual do PDF.
        manter_planilhas (bool): Indica se a planilha física deve ser mantida após o envio.
        exibir_prints (bool): Indica se mensagens informativas devem ser exibidas.
        id_log (str): Identificador para registro de log.

    Returns:
        bool: True se o envio foi bem-sucedido, False caso contrário.
    """
    dic_cids_sem_colch = {dic_mapa_cids[chave]: img_bytes for chave, img_bytes in dic_imagens_originais.items()}
    dic_cids_com_colch = {f"<{dic_mapa_cids[chave]}>": img_bytes for chave, img_bytes in dic_imagens_originais.items()}

    if caminho_pdf:
        gerar_pdf_do_email(
            msg_loja=msg_loja,
            html_email=html_corpo,
            dic_imagens_embutidas=dic_cids_sem_colch,
            eh_notebook=eh_notebook,
            caminho_pdf=caminho_pdf,
            exibir_prints=exibir_prints,
            nome_anexo=nome_anexo_pdf,
            bytes_anexo=bytes_anexo,
        )

    return enviar_email(
        servidor_email=servidor_email,
        assunto=assunto,
        remetente=servidor_email.user,
        destinatario=destinatario,
        html_corpo=html_corpo,
        dic_imagens_embutidas=dic_cids_com_colch,
        caminho_planilha=caminho_planilha,
        bytes_anexo=bytes_anexo,
        nome_anexo=nome_anexo_email,
        manter_planilhas=manter_planilhas,
        exibir_prints=exibir_prints,
        id_log=id_log,
    )


### 6.6. Envio dos e-mails individuais às lojas

Percorre as lojas para gerar seus gráficos, PDFs e e-mails individuais.


In [ ]:
def enviar_emails_lojas(
    servidor_email: smtplib.SMTP,
    nome_remetente: str,
    df_lojas: pd.DataFrame,
    dic_metas: dict,
    df_vendas_consolidadas_lojas: pd.DataFrame,
    data_ontem_str_dma: str,
    data_ontem_str_amd: str,
    template_compilado: Template,
    dic_planilhas_lojas: dict[int, Path],
    caminho_graficos: Path | None,
    caminho_pdfs: Path | None,
    eh_notebook: bool,
    exibir_prints: bool,
    manter_planilhas: bool,
) -> bool:
    """Envia e-mails individuais às lojas com desempenho diário/anual, gráfico mensal e planilha de vendas.

    Args:
        servidor_email (smtplib.SMTP): Servidor de e-mail ativo.
        nome_remetente (str): Nome do remetente.
        df_lojas (pd.DataFrame): Tabela de dados cadastrais das lojas.
        dic_metas (dict): Dicionário com as metas diárias e anuais das lojas.
        df_vendas_consolidadas_lojas (pd.DataFrame): Tabela de vendas consolidadas das lojas.
        data_ontem_str_dma (str): Data do dia anterior em formato dd/mm/aaaa.
        data_ontem_str_amd (str): Data do dia anterior em formato aaaa-mm-dd.
        template_compilado (Template): Template HTML compilado das lojas.
        dic_planilhas_lojas (dict[int, Path]): Dicionário mapeando ID da loja ao caminho da sua planilha.
        caminho_graficos (Path | None): Diretório para salvar imagens dos gráficos ou None.
        caminho_pdfs (Path | None): Diretório para salvar PDFs de simulação ou None.
        eh_notebook (bool): Indicador de ambiente Jupyter Notebook.
        exibir_prints (bool): Indicador de exibição de mensagens informativas.
        manter_planilhas (bool): Indicador de manutenção das planilhas em disco após o envio.

    Returns:
        bool: Sucesso ou falha no envio para todas as lojas.
    """
    try:
        df_lojas_geral = df_vendas_consolidadas_lojas.join(df_lojas[["Loja", "Gerente", "E-mail"]])

        for id_loja, colunas in df_lojas_geral.iterrows():
            id_log = f"loja {id_loja}"
            loja_id_nome = f"{id_loja} - {colunas['Loja']}"
            registrar_log(f"Processando e-mail da loja: {loja_id_nome}", "INFO", exibir_prints)

            caminho_planilha_loja = dic_planilhas_lojas[id_loja]
            df_mensal_loja = pd.read_excel(caminho_planilha_loja, sheet_name="Mensal")

            html_loja, dic_cids_loja = construir_html_email(
                nome_remetente=nome_remetente,
                data_ontem_str_dma=data_ontem_str_dma,
                colunas_dados=colunas,
                dic_metas=dic_metas,
                template=template_compilado,
                lst_cids_imagens=["grafico_vendas"],
                id_log=id_log,
                gerente=colunas["Gerente"],
                nome_loja=colunas["Loja"],
            )
            if html_loja is None:
                return False

            caminho_img_loja = (
                caminho_graficos / f"{data_ontem_str_amd}_Vendas_Mensais_Loja_{id_loja}.png"
                if caminho_graficos
                else None
            )
            img_bytes = construir_grafico_evol_vendas(
                loja_id_nome=loja_id_nome,
                meses=df_mensal_loja["Mês"],
                quantidades=df_mensal_loja["Quantidade"],
                faturamentos=df_mensal_loja["Valor Final"],
                caminho_grafico=caminho_img_loja,
                id_log=id_log,
            )
            if not img_bytes:
                return False

            bytes_planilha = caminho_planilha_loja.read_bytes()
            caminho_pdf_loja = (
                caminho_pdfs / f"{data_ontem_str_amd} - OnePage - Loja {loja_id_nome}.pdf"
                if caminho_pdfs
                else None
            )

            if not _processar_pdf_e_enviar_email(
                servidor_email=servidor_email,
                assunto=f"OnePage Dia {data_ontem_str_dma} - Loja {loja_id_nome}",
                destinatario=colunas["E-mail"],
                html_corpo=html_loja,
                dic_mapa_cids=dic_cids_loja,
                dic_imagens_originais={"grafico_vendas": img_bytes},
                caminho_pdf=caminho_pdf_loja,
                eh_notebook=eh_notebook,
                msg_loja=True,
                caminho_planilha=caminho_planilha_loja,
                bytes_anexo=bytes_planilha,
                nome_anexo_email=caminho_planilha_loja.name,
                nome_anexo_pdf=caminho_planilha_loja.name,
                manter_planilhas=manter_planilhas,
                exibir_prints=exibir_prints,
                id_log=id_log,
            ):
                return False

        return True
    except Exception as erro:
        registrar_log(f"Erro no envio de e-mails às lojas: {erro}!", "CRITICAL")
        return False

### 6.7. Preparação do ranking diário e anual

Cria tabelas ordenadas com metas e resultados para a diretoria.


In [30]:
def _preparar_ranking_lojas(
    df_consolidado: pd.DataFrame,
    dic_metas_lojas: dict,
    dic_nomes_lojas: dict,
    sufixo_col: str,
    sufixo_meta: str,
) -> pd.DataFrame:
    """Ordena, renomeia, adiciona metas e estrutura a tabela de ranking para um período específico.

    Args:
        df_consolidado (pd.DataFrame): Tabela consolidada das lojas.
        dic_metas_lojas (dict): Dicionário com as metas das lojas.
        dic_nomes_lojas (dict): Dicionário mapeando {id_loja: nome_loja}.
        sufixo_col (str): Sufixo das colunas do DataFrame ('Ano' ou 'Ontem').
        sufixo_meta (str): Sufixo das chaves no dicionário de metas ('ano' ou 'dia').

    Returns:
        pd.DataFrame: Tabela de ranking pronta para exportação.
    """
    col_fat = f"Fat_{sufixo_col}"
    
    # Ordena por faturamento (decrescente) e desempata pelo ID da loja (crescente)
    df_rank = df_consolidado.sort_index(ascending=True).sort_values(by=col_fat, ascending=False).copy()

    # Renomeia as colunas do período para os cabeçalhos executivos
    df_rank = df_rank.rename(columns={
        f"Fat_{sufixo_col}": "Faturamento (R$)",
        f"Meta_Fat_{sufixo_col}": "Faturamento Ok",
        f"Prod_Div_{sufixo_col}": "Diversidade de Produtos",
        f"Meta_Div_{sufixo_col}": "Diversidade Ok",
        f"Tic_Medio_{sufixo_col}": "Ticket Médio (R$)",
        f"Meta_Tic_{sufixo_col}": "Ticket Ok",
    })

    # Acrescenta identificação da loja e valores das metas
    df_rank["Loja"] = df_rank.index.astype(str) + " - " + df_rank.index.map(dic_nomes_lojas)
    df_rank["Meta Faturamento (R$)"] = dic_metas_lojas[f"fat_{sufixo_meta}"]
    df_rank["Meta Diversidade"] = dic_metas_lojas[f"div_{sufixo_meta}"]
    df_rank["Meta Ticket (R$)"] = dic_metas_lojas[f"tic_{sufixo_meta}"]

    # Retorna as colunas
    return df_rank[["Loja", "Faturamento (R$)", "Meta Faturamento (R$)", "Faturamento Ok", "Diversidade de Produtos", "Meta Diversidade", "Diversidade Ok", "Ticket Médio (R$)", "Meta Ticket (R$)", "Ticket Ok",]]


### 6.8. Envio do relatório consolidado à diretoria

Monta o relatório consolidado da diretoria, incluindo rankings e gráficos.

In [31]:
def enviar_email_diretoria(
    servidor_email: smtplib.SMTP,
    nome_remetente: str,
    df_vendas: pd.DataFrame,
    dic_metas_lojas: dict,
    dic_metas_rede: dict,
    df_vendas_consolidadas_lojas: pd.DataFrame,
    df_vendas_consolidadas_rede: pd.DataFrame,
    email_diretoria: str,
    data_ontem_str_dma: str,
    data_ontem_str_amd: str,
    template_compilado: Template,
    caminho_graficos: Path | None,
    caminho_pdfs: Path | None,
    eh_notebook: bool,
    dic_nomes_lojas: dict[int, str],
    exibir_prints: bool,
) -> bool:
    """Envia e-mail consolidado à diretoria com indicadores da rede, gráficos e planilha de rankings.

    Args:
        servidor_email (smtplib.SMTP): Servidor de e-mail ativo.
        nome_remetente (str): Nome do remetente.
        df_vendas (pd.DataFrame): Tabela geral de vendas.
        dic_metas_lojas (dict): Dicionário com as metas anuais e diárias das lojas.
        dic_metas_rede (dict): Dicionário com as metas anuais e diárias da rede.
        df_vendas_consolidadas_lojas (pd.DataFrame): Tabela de vendas consolidadas das lojas.
        df_vendas_consolidadas_rede (pd.DataFrame): Tabela de vendas consolidadas de toda a rede.
        email_diretoria (str): Endereço de e-mail da diretoria.
        data_ontem_str_dma (str): Data do dia anterior em formato dd/mm/aaaa.
        data_ontem_str_amd (str): Data do dia anterior em formato aaaa-mm-dd.
        template_compilado (Template): Template HTML compilado da diretoria.
        caminho_graficos (Path | None): Diretório para salvar imagens dos gráficos ou None.
        caminho_pdfs (Path | None): Diretório para salvar PDFs de simulação ou None.
        eh_notebook (bool): Indicador de ambiente Jupyter Notebook.
        dic_nomes_lojas (dict[int, str]): Dicionário de nomes das lojas.
        exibir_prints (bool): Indicador de exibição de mensagens informativas.

    Returns:
        bool: Sucesso ou falha no envio à diretoria.
    """
    try:
        df_ranking_ano = _preparar_ranking_lojas(
            df_vendas_consolidadas_lojas, dic_metas_lojas, dic_nomes_lojas, "Ano", "ano"
        )
        df_ranking_ontem = _preparar_ranking_lojas(
            df_vendas_consolidadas_lojas, dic_metas_lojas, dic_nomes_lojas, "Ontem", "dia"
        )

        fat_ano_max = df_ranking_ano["Faturamento (R$)"].iloc[0]
        lojas_max_ano = df_ranking_ano[df_ranking_ano["Faturamento (R$)"] == fat_ano_max]["Loja"].tolist()
        fat_ano_min = df_ranking_ano["Faturamento (R$)"].iloc[-1]
        lojas_min_ano = df_ranking_ano[df_ranking_ano["Faturamento (R$)"] == fat_ano_min]["Loja"].tolist()

        fat_ontem_max = df_ranking_ontem["Faturamento (R$)"].iloc[0]
        lojas_max_ontem = df_ranking_ontem[df_ranking_ontem["Faturamento (R$)"] == fat_ontem_max]["Loja"].tolist()
        fat_ontem_min = df_ranking_ontem["Faturamento (R$)"].iloc[-1]
        lojas_min_ontem = df_ranking_ontem[df_ranking_ontem["Faturamento (R$)"] == fat_ontem_min]["Loja"].tolist()

        id_log = "diretoria"

        html_diretoria, dic_cids_diretoria = construir_html_email(
            nome_remetente=nome_remetente,
            data_ontem_str_dma=data_ontem_str_dma,
            colunas_dados=df_vendas_consolidadas_rede.iloc[0],
            dic_metas=dic_metas_rede,
            template=template_compilado,
            lst_cids_imagens=["grafico_mensal_rede", "grafico_anual_lojas"],
            id_log=id_log,
            qtd_lojas=len(df_vendas_consolidadas_lojas),
            fat_ano_max=converter_notacao_europeia(fat_ano_max),
            lojas_max_ano=lojas_max_ano,
            fat_ano_min=converter_notacao_europeia(fat_ano_min),
            lojas_min_ano=lojas_min_ano,
            fat_ontem_max=converter_notacao_europeia(fat_ontem_max),
            lojas_max_ontem=lojas_max_ontem,
            fat_ontem_min=converter_notacao_europeia(fat_ontem_min),
            lojas_min_ontem=lojas_min_ontem,
        )
        if html_diretoria is None:
            return False

        df_rede_mes = (
            df_vendas.groupby(df_vendas["Data"].dt.to_period("M"))[["Quantidade", "Valor Final"]]
            .sum()
            .reset_index()
            .rename(columns={"Data": "Mês"})
        )
        df_rede_mes["Mês"] = df_rede_mes["Mês"].astype(str)

        caminho_img_rede = (
            caminho_graficos / f"{data_ontem_str_amd}_Vendas_Mensais_Rede.png" if caminho_graficos else None
        )
        img_bytes_mensal_rede = construir_grafico_evol_vendas(
            loja_id_nome=None,
            meses=df_rede_mes["Mês"],
            quantidades=df_rede_mes["Quantidade"],
            faturamentos=df_rede_mes["Valor Final"],
            caminho_grafico=caminho_img_rede,
            id_log=id_log,
        )
        if not img_bytes_mensal_rede:
            return False

        df_grafico_anual = (
            df_ranking_ano[["Loja", "Faturamento (R$)"]]
            .rename(columns={"Faturamento (R$)": "Fat_Ano"})
            .sort_values(by="Fat_Ano", ascending=True)
            .reset_index(drop=True)
        )
        caminho_img_anual = (
            caminho_graficos / f"{data_ontem_str_amd}_Vendas_Anuais_Lojas.png" if caminho_graficos else None
        )
        img_bytes_anual_lojas = construir_grafico_anual_lojas(df_grafico_anual, caminho_img_anual)
        if not img_bytes_anual_lojas:
            return False

        bytes_df_rede = _converter_df_bytes({"Ranking Anual": df_ranking_ano, "Ranking Diário": df_ranking_ontem})
        caminho_pdf_diretoria = (
            caminho_pdfs / f"{data_ontem_str_amd} - OnePage - Rede de Lojas.pdf" if caminho_pdfs else None
        )

        return _processar_pdf_e_enviar_email(
            servidor_email=servidor_email,
            assunto=f"OnePage Dia {data_ontem_str_dma} - Rede de Lojas",
            destinatario=email_diretoria,
            html_corpo=html_diretoria,
            dic_mapa_cids=dic_cids_diretoria,
            dic_imagens_originais={
                "grafico_mensal_rede": img_bytes_mensal_rede,
                "grafico_anual_lojas": img_bytes_anual_lojas,
            },
            caminho_pdf=caminho_pdf_diretoria,
            eh_notebook=eh_notebook,
            msg_loja=False,
            caminho_planilha=None,
            bytes_anexo=bytes_df_rede,
            nome_anexo_email=f"{data_ontem_str_amd} - Ranking_Vendas_Lojas.xlsx",
            nome_anexo_pdf="Ranking_Vendas_Lojas.xlsx",
            manter_planilhas=True,
            exibir_prints=exibir_prints,
            id_log=id_log,
        )
    except Exception as erro:
        registrar_log(f"Erro no envio de e-mail à diretoria: {erro}!", "CRITICAL")
        return False

## 7. Execução principal da automação

Coordena o fluxo completo de ponta a ponta utilizando uma data de referência única capturada no início da execução (evitando divergências caso o processo atravesse a meia-noite). Garante o encerramento seguro da conexão SMTP através de um bloco `try...finally`.


In [32]:
def main():
    """Coordena todo o fluxo de processamento de dados, geração de relatórios e envio de e-mails."""
    agora = pd.Timestamp.now()
    data_hoje = agora.normalize()
    data_ontem = data_hoje - pd.Timedelta(days=1)

    data_hoje_str_amd = data_hoje.strftime("%Y-%m-%d")
    data_ontem_str_amd = data_ontem.strftime("%Y-%m-%d")
    data_ontem_str_dma = data_ontem.strftime("%d/%m/%Y")

    print(f"{agora.strftime('%H:%M:%S')} - Identificação do ambiente de execução")
    eh_notebook, caminho_raiz = identificar_ambiente()
    if not caminho_raiz:
        return
    
    config = carregar_configuracoes(eh_notebook, caminho_raiz)
    exibir_prints = config["outputs"]["exibir_prints_acompanhamento"]
    manter_planilhas = config["outputs"]["manter_planilhas"]
    exportar_graficos = config["outputs"]["exportar_graficos"]
    gerar_pdfs = config["outputs"]["gerar_pdfs_email"]

    caminho_planilhas = caminho_raiz / config["caminhos"]["saida_planilhas"]
    caminho_graficos = caminho_raiz / config["caminhos"]["saida_graficos"] if exportar_graficos else None
    caminho_pdfs = caminho_raiz / config["caminhos"]["saida_pdfs"] if gerar_pdfs else None

    if exibir_prints:
        print(f"{agora.strftime('%H:%M:%S')} - Configuração do log")
    if not configurar_log(caminho_raiz, data_hoje_str_amd):
        return

    registrar_log("Início geral da execução", "INFO", exibir_prints)

    registrar_log("Busca das variáveis de ambiente", "INFO", exibir_prints)
    credenciais = buscar_variaveis_ambiente(caminho_raiz / ".env")
    if not credenciais:
        return

    registrar_log("Preparação das bases de dados", "INFO", exibir_prints)
    bases = preparar_bases_processadas(caminho_raiz, caminho_planilhas, credenciais.destinatario_teste, data_ontem)
    if not bases:
        return

    registrar_log("Unificação dos dados gerais das lojas", "INFO", exibir_prints)
    df_lojas, dic_nomes_lojas, email_diretoria = unificar_dados_lojas(bases.caminho_raw / "Lojas.csv", bases.df_emails)
    if df_lojas is None:
        return

    registrar_log("Geração dos valores consolidados", "INFO", exibir_prints)
    df_vendas_lojas, df_vendas_rede = gerar_valores_consolidados(bases.df_vendas, data_ontem)
    if df_vendas_lojas is None:
        return

    registrar_log("Cálculo de atingimento de metas", "INFO", exibir_prints)
    metas = preencher_atingimento_metas(bases.df_metas_lojas, df_vendas_lojas, df_vendas_rede)
    if not metas:
        return

    registrar_log("Configuração do ambiente Jinja", "INFO", exibir_prints)
    env_jinja, nome_template_lojas, nome_template_diretoria = configurar_ambiente_jinja(caminho_raiz)
    if not env_jinja:
        return

    registrar_log("Conexão com o servidor de e-mail", "INFO", exibir_prints)
    servidor_email = conectar_servidor_smtp(
        credenciais.remetente, credenciais.senha, credenciais.porta, credenciais.servidor
    )
    if not servidor_email:
        return

    if caminho_graficos:
        caminho_graficos.mkdir(parents=True, exist_ok=True)
    if caminho_pdfs:
        caminho_pdfs.mkdir(parents=True, exist_ok=True)

    try:
        registrar_log("Envio de e-mails às lojas", "INFO", exibir_prints)
        sucesso_lojas = enviar_emails_lojas(
            servidor_email=servidor_email,
            nome_remetente=credenciais.nome_remetente,
            df_lojas=df_lojas,
            dic_metas=metas.dic_metas_lojas,
            df_vendas_consolidadas_lojas=metas.df_vendas_lojas,
            data_ontem_str_dma=data_ontem_str_dma,
            data_ontem_str_amd=data_ontem_str_amd,
            template_compilado=env_jinja.get_template(nome_template_lojas),
            dic_planilhas_lojas=bases.dic_planilhas_lojas,
            caminho_graficos=caminho_graficos,
            caminho_pdfs=caminho_pdfs,
            eh_notebook=eh_notebook,
            exibir_prints=exibir_prints,
            manter_planilhas=manter_planilhas,
        )
        if not sucesso_lojas:
            return

        registrar_log("Envio de e-mail à diretoria com rankings das lojas", "INFO", exibir_prints)
        enviar_email_diretoria(
            servidor_email=servidor_email,
            nome_remetente=credenciais.nome_remetente,
            df_vendas=bases.df_vendas,
            dic_metas_lojas=metas.dic_metas_lojas,
            dic_metas_rede=metas.dic_metas_rede,
            df_vendas_consolidadas_lojas=metas.df_vendas_lojas,
            df_vendas_consolidadas_rede=metas.df_vendas_rede,
            email_diretoria=email_diretoria,
            data_ontem_str_dma=data_ontem_str_dma,
            data_ontem_str_amd=data_ontem_str_amd,
            template_compilado=env_jinja.get_template(nome_template_diretoria),
            caminho_graficos=caminho_graficos,
            caminho_pdfs=caminho_pdfs,
            eh_notebook=eh_notebook,
            dic_nomes_lojas=dic_nomes_lojas,
            exibir_prints=exibir_prints,
        )
    finally:
        registrar_log("Encerramento da conexão com o servidor de e-mail", "INFO", exibir_prints)
        desconectar_servidor_smtp(servidor_email, exibir_prints)
        registrar_log("Finalização da execução", "INFO", exibir_prints)
        encerrar_log()

In [33]:
if __name__ == "__main__":
    main()

20:43:19 - Identificação do ambiente de execução
20:43:19 - Configuração do log
20:43:19 - Início geral da execução
20:43:19 - Busca das variáveis de ambiente
20:43:19 - Preparação das bases de dados
20:43:33 - Unificação dos dados gerais das lojas
20:43:33 - Geração dos valores consolidados
20:43:33 - Cálculo de atingimento de metas
20:43:33 - Configuração do ambiente Jinja
20:43:33 - Conexão com o servidor de e-mail
20:43:35 - Envio de e-mails às lojas
20:43:35 - Processando e-mail da loja: 1 - Iguatemi Esplanada
20:43:44 - E-mail para loja 1 enviado com sucesso!
20:43:44 - Processando e-mail da loja: 2 - Shopping Midway Mall
20:43:51 - E-mail para loja 2 enviado com sucesso!
20:43:51 - Envio de e-mail à diretoria com rankings das lojas
20:44:00 - E-mail para diretoria enviado com sucesso!
20:44:00 - Encerramento da conexão com o servidor de e-mail
20:44:00 - Finalização da execução
